In [18]:
import re
import warnings

import nltk
import numpy as np
import pandas as pd
from tqdm import tqdm

warnings.filterwarnings('ignore')

tqdm.pandas(desc='processing')
nltk.download('stopwords', quiet=True)

import pymorphy3

from nltk.corpus import stopwords
from nltk.stem.snowball import SnowballStemmer

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.preprocessing import LabelEncoder
from sklearn.svm import LinearSVC


In [19]:

# ── Split ──────────────────────────────────────────────────────────────────
RANDOM_STATE = 42
FIXATED_TEST_SIZE = 5000
MIN_TRAIN_PER_CLASS = 3000 
MAX_TRAIN_PER_CLASS = 5000

# ── NLP tools ──────────────────────────────────────────────────────────────
RU_STOPWORDS = set(stopwords.words('russian'))
MORPH        = pymorphy3.MorphAnalyzer()
STEMMER      = SnowballStemmer('russian')

DOMAIN_STOPWORDS = {
    'психолог', 'психотерапевт', 'специалист', 'терапия', 'терапевт',
    'психотерапия', 'психиатр', 'психиатрия',
    'человек', 'люди', 'жизнь', 'время', 'ситуация', 'проблема',
    'помощь', 'помочь', 'работа', 'работать',
    'автор', 'пост', 'форум', 'комментарий', 'тема',
    'вообще', 'просто', 'конечно', 'наверное', 'наверно',
    'кстати', 'вроде', 'типа', 'короче',
    'это', 'очень'
}
RU_STOPWORDS |= DOMAIN_STOPWORDS

np.random.seed(RANDOM_STATE)

In [20]:

# Load, Preprocess Entire Dataset & Split
print("\n[1/6] Loading data and selecting columns...")

# ── Paths ──────────────────────────────────────────────────────────────────
DATA_PATH = 'data.csv'
AUGMENTED_DATA_PATH = 'augmented_data.csv'

# Load initial data
df_initial = pd.read_csv(DATA_PATH, sep=';', encoding='utf-8')
print(f"Initial data shape: {df_initial.shape}  |  columns: {list(df_initial.columns)}")

# Keep only the two required columns
df_initial = df_initial[['text', 'tag']].copy()
df_initial = df_initial.dropna(subset=['text', 'tag']).reset_index(drop=True)
df_initial['text'] = df_initial['text'].astype(str)
df_initial['tag']  = df_initial['tag'].astype(str).str.strip()

# Drop ambiguous / noisy classes from initial data
before = len(df_initial)
df_initial = df_initial[~df_initial['tag'].isin(CLASSES_TO_DROP)].copy().reset_index(drop=True)
print(f"After dropping ambiguous classes: removed {before - len(df_initial):,} -> {len(df_initial):,} rows, "
      f"{df_initial['tag'].nunique()} classes")

print("\nClass distribution (initial data):")
print(df_initial['tag'].value_counts().to_string())


[1/6] Loading data and selecting columns...
Initial data shape: (66460, 5)  |  columns: ['url', 'date', 'text', 'tag', 'source']
After dropping ambiguous classes: removed 3,123 -> 63,336 rows, 6 classes

Class distribution (initial data):
tag
депрессия         37639
тревожное р-во    10069
ОКР                5393
ПРЛ                5219
БАР                2961
шизофрения         2055


In [21]:
# Preprocess the entire dataset once

_URL_RE    = re.compile(r'https?://\S+|www\.\S+')
_EMAIL_RE  = re.compile(r'\S+@\S+\.\S+')
_HTML_RE   = re.compile(r'<[^>]+>')
_MENTION_RE= re.compile(r'@\w+')           # @username mentions
_HASHTAG_RE= re.compile(r'#\w+')           # #hashtags
_DIGIT_RE  = re.compile(r'\b\d+\b')        # standalone numbers
_PUNCT_RE  = re.compile(r'[^а-яёА-ЯЁa-zA-Z\s]')  # keep only letters + spaces
_WS_RE     = re.compile(r'\s+')
# Collapse 3+ repeated characters to 2: "оооочень" → "оочень"
_REPEAT_RE = re.compile(r'(.)\1{2,}')

_GREETING_RE = re.compile(
    r'^\s*(?:здравствуйте[!,.]?\s*|добрый\s+\w+[!,.]?\s*'
    r'|доброго\s+\w+(?:\s+\w+)?[!,.]?\s*|привет[!,.]?\s*)+',
    re.I,
)
_SIGNOFF_RE = re.compile(
    r'(?:спасибо\s+за\s+(?:ваш\s+)?ответ|с\s+уважением[\s\w,]*'
    r'|всего\s+доброго|удачи\s+вам|хорошего\s+дня)[.!]?\s*$',
    re.I,
)
_META_AUTHOR_RE = re.compile(r'^\s*автор\s*,\s*', re.I)
_BOOK_RE = re.compile(
    r'(?:рекомендую\s+(?:книгу|почитать)|книга\s+[«\"][^»\"]{1,60}[»\"]'
    r'|читайте\s+книгу)',
    re.I,
)
_PROMO_RE = re.compile(
    r'(?:подписывайтесь(?:\s+на\s+(?:мой|наш)\s+\w+)?'
    r'|мой\s+(?:канал|блог|телеграм|telegram)'
    r'|t\.me/\S+|instagram\.com/\S+|vk\.com/\S+)',
    re.I,
)

# Negation particles that flip the meaning of the following word
_NEGATION_PARTICLES = {'не', 'ни'}


def filter_text(text: str) -> str:
    text = _URL_RE.sub(' ', text)
    text = _EMAIL_RE.sub(' ', text)
    text = _HTML_RE.sub(' ', text)
    text = _MENTION_RE.sub(' ', text)
    text = _HASHTAG_RE.sub(' ', text)
    text = _DIGIT_RE.sub(' ', text)
    text = _GREETING_RE.sub(' ', text)
    text = _SIGNOFF_RE.sub(' ', text)
    text = _META_AUTHOR_RE.sub(' ', text)
    text = _BOOK_RE.sub(' ', text)
    text = _PROMO_RE.sub(' ', text)
    text = _PUNCT_RE.sub(' ', text)
    text = _REPEAT_RE.sub(r'\1\1', text)
    text = _WS_RE.sub(' ', text).strip()
    tokens = text.split()
    if len(tokens) > MAX_WORDS:
        tokens = tokens[:MAX_WORDS]
    return ' '.join(tokens)

In [22]:
def remove_stopwords_and_normalize(text: str) -> str:
    tokens = text.split()
    tokens = [t for t in tokens if t.lower() not in RU_STOPWORDS]
    result = []
    for t in tokens:
        if t.startswith('не_'):
            word = t[3:]
            lemma = MORPH.parse(word)[0].normal_form if word else word
            stem  = STEMMER.stem(lemma) if lemma else lemma
            result.append('не_' + stem)
        else:
            lemma = MORPH.parse(t)[0].normal_form
            result.append(STEMMER.stem(lemma))
    tokens = [t for t in result if len(t) >= 2]
    return ' '.join(tokens)


In [23]:


CLASS_MARKERS = {
    'депрессия': [
        'депресси', 'депрессив', 'антидепрессант', 'ангедони',
        'флуоксетин', 'сертралин', 'эсциталопрам', 'венлафаксин', 'миртазапин',
        'апати', 'тоск', 'безнадёжн', 'безнадежн', 'опустошён', 'опустошен',
        'суицид', 'суицидальн', 'самоубийств',
        'нет сил', 'без сил', 'нет энергии', 'нет желания',
        'плохо сплю', 'просыпаюсь ночью', 'бессонниц', 'гиперсомни',
        'прокрастинац', 'выгорани', 'лежу',
        'всё бессмысленн', 'ненавижу себя', 'я обуза',
        'потерял интерес', 'нет удовольствия', 'потерял смысл',
        'заторможен', 'вялост', 'истощен',
    ],
    'тревожное р-во': [
        'паническ', 'паник', 'тревог', 'тревожн', 'гтр', 'птср',
        'агорафоби', 'социофоби', 'фоби', 'невроз',
        'социальн тревог', 'генерализованн тревог',
        'страх', 'боюсь', 'боится', 'беспокойств', 'волнени',
        'навязчив', 'избегани', 'катастрофизац',
        'предчувствие беды', 'ожидание худшего',
        'сердце бьёт', 'сердце бьет', 'сердцебиени', 'учащённ пульс',
        'задыхаюсь', 'удушь', 'дрожь', 'трясёт', 'потею',
        'холодный пот', 'головокружени', 'живот сводит', 'мышцы напряжен',
        'боюсь выходить', 'тремор', 'гипервентиляц',
        'вегетатив', 'дереализаци', 'деперсонализаци',
        'грандаксин', 'феназепам', 'атаракс', 'афобазол', 'когнитивно поведенческ',
    ],
    'ОКР': [
        'навязчив', 'компульси', 'обсесси', 'окр', 'оkр', 'навязчивост',
        'ритуал', 'ритуальн', 'перепроверк', 'проверяю', 'протираю',
        'дезинфицирую', 'повторяю', 'считаю', 'пересчитыв',
        'мою руки', 'мою посуду',
        'стерильн', 'страх заражени', 'страх загрязнени', 'страх причинить',
        'симметри', 'магическ мышлени', 'навязчивые мысли',
        'нежелательн мысл', 'агрессивн навязчив', 'богохульн мысл',
        'перфекционизм', 'иррациональн',
        'флувоксамин', 'кломипрамин', 'анафранил',
    ],
    'ПРЛ': [
        'пустот', 'пограничн', 'прл', 'бпр',
        'идеализаци', 'обесценивани', 'расщеплени', 'черно-белое мышлени',
        'страх одиночеств', 'бурн отношени', 'токсичн отношени',
        'размытая идентичность',
        'эмоциональн качел', 'импульсивн', 'вспышки гнева',
        'резкие перепады', 'захлёстывает', 'нестабильн',
        'самоповреждени', 'порезы', 'членовредительств', 'режу себя',
        'диссоциаци', 'деперсонализаци',
        'диалектическ', 'дбт', 'dbt',
    ],
    'БАР': [
        'биполяр', 'бар ', ' бар', 'мани', 'гипомани', 'нормотимик', 'циклотими',
        'литий', 'депакин', 'ламиктал', 'ламотриджин',
        'сероквел', 'кветиапин', 'карбамазепин', 'вальпроат', 'тегретол',
        'смена настроени', 'перепады настроени', 'цикл настроени',
        'депрессивн эпизод', 'смешанн эпизод',
        'фаза подъёма', 'фаза спада', 'цикличн',
        'эйфори', 'грандиозн', 'маниакальн', 'не сплю', 'бессонниц',
        'ускоренн мышлени', 'мысли скачут', 'грандиозные планы',
        'трачу деньги', 'безрассудн', 'гиперсексуальн',
    ],
    'шизофрения': [
        'шизо', 'шизотип', 'шизоаффектив', 'параноидн', 'паранои',
        'галлюцинаци', 'галлюцинир', 'слышу голоса', 'голоса',
        'бред', 'бредов', 'бред преследовани', 'бред величи',
        'психоз', 'психотич', 'острый психоз',
        'алоги', 'абули', 'уплощённ аффект',
        'негативн симптом', 'позитивн симптом',
        'разорванн мышлени', 'расщеплени личност',
        'деперсонализаци', 'дереализаци', 'кататони',
        'антипсихотик', 'нейролептик',
        'галоперидол', 'клозапин', 'оланзапин', 'рисперидон', 'арипипразол',
        'зипрасидон', 'палиперидон', 'амисульприд', 'кветиапин',
        'психиатрическ больниц', 'стационар', 'принудительн лечени',
    ],
}

RARE_CLASSES = {'БАР', 'ОКР', 'ПРЛ'}
RARE_MIN_WORDS = 20

def _normalize_marker(phrase: str) -> str:
    """Lemmatize + stem each word in a marker phrase (same pipeline as the main text)."""
    tokens = phrase.split()
    result = []
    for t in tokens:
        lemma = MORPH.parse(t)[0].normal_form
        result.append(STEMMER.stem(lemma))
    return ' '.join(result)


CLASS_MARKERS_PROCESSED: dict[str, list[str]] = {
    tag: [_normalize_marker(m) for m in markers]
    for tag, markers in CLASS_MARKERS.items()
}


def is_quality_text(text_processed: str, tag: str) -> bool:
    """Keep text if it has ≥ 1 class marker (in processed token space),
    or is from a rare class with enough tokens."""
    if not isinstance(text_processed, str):
        return False
    if tag not in CLASS_MARKERS_PROCESSED:
        return True
    tokens = set(text_processed.split())
    found = sum(
        1 for phrase in CLASS_MARKERS_PROCESSED[tag]
        if all(t in tokens for t in phrase.split())
    )
    return found >= 1 or (tag in RARE_CLASSES and len(tokens) >= RARE_MIN_WORDS)

In [24]:
# ── Filtering thresholds ───────────────────────────────────────────────────
MIN_WORDS    = 3 
MAX_WORDS    = 1000

# ── Classes to drop (ambiguous / too noisy) ────────────────────────────────
CLASSES_TO_DROP = ['тревожное р-во/депрессия', 'паранойя', 'тревожное р-во/невроз']


def preprocess_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    """Apply full text preprocessing pipeline to a dataframe with 'text' and 'tag' columns.
    Returns a new dataframe with 'text_clean', 'text_processed', 'word_count' columns added."""
    df = df.copy()

    tqdm.pandas(desc='filter_text')
    df['text_clean'] = df['text'].progress_apply(filter_text)

    tqdm.pandas(desc='stopwords+lemmatize+stem')
    df['text_processed'] = df['text_clean'].progress_apply(remove_stopwords_and_normalize)

    df['word_count'] = df['text_processed'].apply(lambda s: len(s.split()))
    before = len(df)
    df = df[df['word_count'] >= MIN_WORDS].copy().reset_index(drop=True)
    print(f"After min-words filter ({MIN_WORDS}): removed {before - len(df):,} -> {len(df):,} kept")

    before = len(df)
    df = df.drop_duplicates(subset=['text_processed', 'tag']).reset_index(drop=True)
    print(f"After de-duplication:               removed {before - len(df):,} -> {len(df):,} kept")

    if 'tag' in df.columns:
        df['_quality'] = df.apply(
            lambda r: is_quality_text(r['text_processed'], r['tag']), axis=1
        )
        q_counts   = df[df['_quality']]['tag'].value_counts()
        all_counts = df['tag'].value_counts()
        before = len(df)
        df = df[df['_quality']].drop(columns=['_quality']).reset_index(drop=True)
        print(f"After quality filter:               removed {before - len(df):,} -> {len(df):,} kept")
        print("Per-class quality keep rate:")
        for tag in all_counts.index:
            kept  = q_counts.get(tag, 0)
            total = all_counts[tag]
            print(f"  {tag:30s}: kept {kept:5,}/{total:5,}  ({kept/total*100:.1f}%)")

    return df

print("\n[2/6] Preprocessing entire dataset...")
df_initial = preprocess_dataframe(df_initial)

print(f"\nWord count stats after processing:")
print(df_initial['word_count'].describe().to_string())

print("\n=== 5 random preprocessed samples ===")
for _, row in df_initial.sample(n=5, random_state=RANDOM_STATE).iterrows():
    tokens = row['text_processed'].split()
    print(f"\n  [tag: {row['tag']}]")
    print(f"  RAW:    {row['text'][:120]}")
    print(f"  TOKENS: {tokens}")
print("=" * 64)

print(f"\nClass distribution after preprocessing:")
print(df_initial['tag'].value_counts().to_string())


[2/6] Preprocessing entire dataset...


stopwords+lemmatize+stem: 100%|██████████| 63336/63336 [01:27<00:00, 725.14it/s] 


After min-words filter (3): removed 6,981 -> 56,355 kept
After de-duplication:               removed 225 -> 56,130 kept
After quality filter:               removed 43,370 -> 12,760 kept
Per-class quality keep rate:
  депрессия                     : kept 4,409/33,324  (13.2%)
  тревожное р-во                : kept 3,078/9,015  (34.1%)
  ОКР                           : kept 2,317/4,848  (47.8%)
  ПРЛ                           : kept 1,877/4,582  (41.0%)
  БАР                           : kept   986/2,581  (38.2%)
  шизофрения                    : kept    93/1,780  (5.2%)

Word count stats after processing:
count    12760.000000
mean        50.768417
std         67.872347
min          3.000000
25%         17.000000
50%         30.000000
75%         56.000000
max        751.000000

=== 5 random preprocessed samples ===

  [tag: тревожное р-во]
  RAW:    Месяц назад были почечные колики у меня впервые, проходила обследование. Это все меня и подкосило, очень переживала. И в
  TOKENS: ['месяц'

In [25]:
# Fixated Test Set Selection
print(f"\n[3/6] Selecting fixated test set ({FIXATED_TEST_SIZE} samples)...")

df_test_fixated_processed = df_initial.sample(n=FIXATED_TEST_SIZE, random_state=RANDOM_STATE)
df_train_initial = df_initial.drop(df_test_fixated_processed.index).reset_index(drop=True)
df_test_fixated_processed = df_test_fixated_processed.reset_index(drop=True)

print(f"Fixated test set: {len(df_test_fixated_processed):,} samples")
print(f"Initial train set: {len(df_train_initial):,} samples")

print("\nFixated test set class distribution:")
print(df_test_fixated_processed['tag'].value_counts().to_string())

print("\nInitial train set class distribution:")
print(df_train_initial['tag'].value_counts().to_string())

df_source = df_train_initial.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
print(f"\nTraining data: {len(df_source):,} rows, {df_source['tag'].nunique()} classes")



[3/6] Selecting fixated test set (5000 samples)...
Fixated test set: 5,000 samples
Initial train set: 7,760 samples

Fixated test set class distribution:
tag
депрессия         1739
тревожное р-во    1180
ОКР                926
ПРЛ                742
БАР                379
шизофрения          34

Initial train set class distribution:
tag
депрессия         2670
тревожное р-во    1898
ОКР               1391
ПРЛ               1135
БАР                607
шизофрения          59

Training data: 7,760 rows, 6 classes


In [26]:
# Augmentation & Upsampling

USE_AUGMENTATION = True   # set to False to skip augmentation entirely

def augment_training_data(df_train: pd.DataFrame, augment_path: str) -> pd.DataFrame:
    """
    Augment training data with additional samples from augmented_data.csv.
    The augmented data is preprocessed before merging so it matches the
    already-preprocessed training data.
    Returns combined dataframe with original + augmented data.
    """
    print(f"\n[AUGMENT] Loading augmented data from {augment_path}...")

    try:
        df_aug = pd.read_csv(augment_path, sep=';', encoding='utf-8')
        print(f"Augmented data shape: {df_aug.shape}")

        # Keep only required columns
        if 'text' in df_aug.columns and 'tag' in df_aug.columns:
            df_aug = df_aug[['text', 'tag']].copy()
        else:
            raise ValueError("Augmented data must contain 'text' and 'tag' columns")

        # Clean augmented data
        df_aug = df_aug.dropna(subset=['text', 'tag']).reset_index(drop=True)
        df_aug['text'] = df_aug['text'].astype(str)
        df_aug['tag'] = df_aug['tag'].astype(str).str.strip()

        # Drop ambiguous classes from augmented data
        df_aug = df_aug[~df_aug['tag'].isin(CLASSES_TO_DROP)].copy().reset_index(drop=True)

        # Preprocess augmented data so it matches the already-preprocessed training data
        print("Preprocessing augmented data...")
        df_aug = preprocess_dataframe(df_aug)

        print(f"After cleaning & preprocessing augmented data: {len(df_aug):,} rows")
        print("Augmented data class distribution:")
        print(df_aug['tag'].value_counts().to_string())

        # Combine original and augmented data
        df_combined = pd.concat([df_train, df_aug], ignore_index=True)
        df_combined = df_combined.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

        print(f"\nCombined training data: {len(df_combined):,} rows "
              f"(original: {len(df_train):,} + augmented: {len(df_aug):,})")

        return df_combined

    except Exception as e:
        print(f"[WARNING] Failed to load augmented data: {e}")
        print("[WARNING] Continuing with original training data only.")
        return df_train




def downsample_train_set(df_train: pd.DataFrame, max_per_class: int,
                         random_state: int = 42) -> pd.DataFrame:
    """
    Downsample training set so that no class has more than max_per_class instances.
    Uses random sampling without replacement for classes that exceed the limit.
    """
    print(f"\n[DOWNSAMPLE] Downsampling training set to max {max_per_class} per class...")

    class_counts = df_train['tag'].value_counts()
    print("Current training set class distribution:")
    print(class_counts.to_string())

    downsampled_dfs = []

    for class_name in df_train['tag'].unique():
        class_df = df_train[df_train['tag'] == class_name]
        current_count = len(class_df)

        if current_count > max_per_class:
            print(f"  Downsampling class '{class_name}': {current_count} -> {max_per_class} (-{current_count - max_per_class})")
            downsampled_class = class_df.sample(n=max_per_class, replace=False, random_state=random_state)
        else:
            downsampled_class = class_df.copy()
            print(f"  Class '{class_name}': {current_count} (no downsampling needed)")

        downsampled_dfs.append(downsampled_class)

    df_downsampled = pd.concat(downsampled_dfs, ignore_index=True)
    df_downsampled = df_downsampled.sample(frac=1, random_state=random_state).reset_index(drop=True)

    print(f"\nDownsampled training set: {len(df_downsampled):,} samples")
    print("Class distribution after downsampling:")
    print(df_downsampled['tag'].value_counts().to_string())

    return df_downsampled


def upsample_train_set(df_train: pd.DataFrame, min_per_class: int,
                       random_state: int = 42) -> pd.DataFrame:
    """
    Upsample training set to ensure each class has at least min_per_class instances.
    Uses random sampling with replacement for classes that need more samples.
    """
    print(f"\n[UPSAMPLE] Upsampling training set to min {min_per_class} per class...")

    class_counts = df_train['tag'].value_counts()
    print("Current training set class distribution:")
    print(class_counts.to_string())

    upsampled_dfs = []

    for class_name in df_train['tag'].unique():
        class_df = df_train[df_train['tag'] == class_name]
        current_count = len(class_df)

        if current_count < min_per_class:
            # Need to upsample this class
            needed = min_per_class - current_count
            print(f"  Upsampling class '{class_name}': {current_count} -> {min_per_class} (+{needed})")

            # Sample with replacement to get additional samples
            additional_samples = class_df.sample(n=needed, replace=True, random_state=random_state)
            upsampled_class = pd.concat([class_df, additional_samples], ignore_index=True)
        else:
            # No upsampling needed
            upsampled_class = class_df.copy()
            print(f"  Class '{class_name}': {current_count} (no upsampling needed)")

        upsampled_dfs.append(upsampled_class)

    df_upsampled = pd.concat(upsampled_dfs, ignore_index=True)
    df_upsampled = df_upsampled.sample(frac=1, random_state=random_state).reset_index(drop=True)

    print(f"\nUpsampled training set: {len(df_upsampled):,} samples")
    print("Final training set class distribution:")
    print(df_upsampled['tag'].value_counts().to_string())

    return df_upsampled

# Apply augmentation if requested (augmented data will be preprocessed inside)
if USE_AUGMENTATION:
    df_source = augment_training_data(df_source, AUGMENTED_DATA_PATH)

# Downsample training set to cap classes at MAX_TRAIN_PER_CLASS
df_train_final = downsample_train_set(df_source, MAX_TRAIN_PER_CLASS, RANDOM_STATE)

# Upsample training set to ensure minimum samples per class
df_train_final = upsample_train_set(df_train_final, MIN_TRAIN_PER_CLASS, RANDOM_STATE)

print(f"\nFinal train set: {len(df_train_final):,} samples")
print(f"Final test set: {len(df_test_fixated_processed):,} samples")

# Encode labels
le = LabelEncoder()
y_train  = le.fit_transform(df_train_final['tag'].to_numpy())
y_test   = le.transform(df_test_fixated_processed['tag'].to_numpy())  # Use same encoder
X_train  = df_train_final['text_processed'].to_numpy(dtype=object)
X_test   = df_test_fixated_processed['text_processed'].to_numpy(dtype=object)
NUM_CLASSES = len(le.classes_)

print(f"Classes ({NUM_CLASSES}): {list(le.classes_)}")
print("Train class counts:",
      dict(zip(le.classes_, np.bincount(y_train, minlength=NUM_CLASSES))))
print("Test  class counts:",
      dict(zip(le.classes_, np.bincount(y_test,  minlength=NUM_CLASSES))))


[AUGMENT] Loading augmented data from augmented_data.csv...
Augmented data shape: (11805, 2)
Preprocessing augmented data...


stopwords+lemmatize+stem: 100%|██████████| 11805/11805 [00:15<00:00, 769.83it/s]


After min-words filter (3): removed 0 -> 11,805 kept
After de-duplication:               removed 0 -> 11,805 kept
After quality filter:               removed 2,368 -> 9,437 kept
Per-class quality keep rate:
  шизофрения                    : kept 1,227/2,000  (61.4%)
  БАР                           : kept 1,929/1,995  (96.7%)
  тревожное р-во                : kept 1,370/1,990  (68.8%)
  ОКР                           : kept 1,854/1,985  (93.4%)
  депрессия                     : kept 1,293/1,940  (66.6%)
  ПРЛ                           : kept 1,764/1,895  (93.1%)
After cleaning & preprocessing augmented data: 9,437 rows
Augmented data class distribution:
tag
БАР               1929
ОКР               1854
ПРЛ               1764
тревожное р-во    1370
депрессия         1293
шизофрения        1227

Combined training data: 17,197 rows (original: 7,760 + augmented: 9,437)

[DOWNSAMPLE] Downsampling training set to max 5000 per class...
Current training set class distribution:
tag
депрессия     

In [27]:


# TF-IDF Feature Extraction
print("\n[5/6] Extracting TF-IDF features...")

TFIDF_PARAMS = dict(
    analyzer = 'word',
    ngram_range = (1, 2),
    max_features= 50000,
    sublinear_tf= True,
    min_df      = 3,
    max_df      = 0.85,
)
tfidf = TfidfVectorizer(**TFIDF_PARAMS)

with tqdm(total=2, desc='TF-IDF', unit='step',
          bar_format='{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}] {postfix}') as pbar:
    pbar.set_description('TF-IDF fit_transform (train)')
    X_train_tfidf = tfidf.fit_transform(X_train)
    pbar.set_postfix(shape=str(X_train_tfidf.shape))
    pbar.update(1)

    pbar.set_description('TF-IDF transform (test)')
    X_test_tfidf = tfidf.transform(X_test)
    pbar.set_postfix(shape=str(X_test_tfidf.shape))
    pbar.update(1)

print(f"TF-IDF matrix -- train: {X_train_tfidf.shape}  |  test: {X_test_tfidf.shape}")
print(f"Vocabulary size: {len(tfidf.vocabulary_):,}")


[5/6] Extracting TF-IDF features...


TF-IDF transform (test): 100%|██████████| 2/2 [00:01] , shape=(5000, 50000)      

TF-IDF matrix -- train: (19476, 50000)  |  test: (5000, 50000)
Vocabulary size: 50,000


In [28]:


# Classification & Evaluation
print("\n[6/6] Training classifiers and evaluating...")

results = {}  # label -> (f1_macro, accuracy, y_pred)

SVC_CS = [0.1, 0.2, 0.3, 0.5, 1.0, 1.5, 2.0, 3.0]

# ── Helper for sklearn classifiers ─────────────────────────────────────────
def evaluate(label: str, clf, pbar: tqdm) -> None:
    """Fit clf on train TF-IDF, predict on test, store and print metrics."""
    pbar.set_description(f'fitting  {label}')
    clf.fit(X_train_tfidf, y_train)
    pbar.set_description(f'predict  {label}')
    y_pred = clf.predict(X_test_tfidf)
    f1     = f1_score(y_test, y_pred, average='macro')
    acc    = accuracy_score(y_test, y_pred)
    results[label] = (f1, acc, y_pred)
    pbar.set_postfix(F1=f'{f1:.4f}', Acc=f'{acc:.4f}')
    pbar.update(1)
    print(f"  {label:<35s} | F1={f1:.4f}  Acc={acc:.4f}")


# ── LinearSVC sweep ────────────────────────────────────────────────────────
svc_specs = [
    (f'LinearSVC  C={C}',
     LinearSVC(C=C,
               max_iter=8000,
               class_weight='balanced',
               random_state=RANDOM_STATE))
    for C in SVC_CS
]

with tqdm(total=len(svc_specs), desc='LinearSVC', unit='model',
          bar_format='{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}] {postfix}') as pbar:
    for label, clf in svc_specs:
        evaluate(label, clf, pbar)


[6/6] Training classifiers and evaluating...


fitting  LinearSVC  C=0.2:  12%|█▎        | 1/8 [00:00<00:01] , Acc=0.7986, F1=0.7256

  LinearSVC  C=0.1                    | F1=0.7256  Acc=0.7986


fitting  LinearSVC  C=0.3:  25%|██▌       | 2/8 [00:00<00:00] , Acc=0.8040, F1=0.7340

  LinearSVC  C=0.2                    | F1=0.7340  Acc=0.8040


...

  LinearSVC  C=1.5                    | F1=0.7357  Acc=0.8062


fitting  LinearSVC  C=3.0:  88%|████████▊ | 7/8 [00:01<00:00] , Acc=0.8032, F1=0.7294

  LinearSVC  C=2.0                    | F1=0.7294  Acc=0.8032


predict  LinearSVC  C=3.0: 100%|██████████| 8/8 [00:02<00:00] , Acc=0.8008, F1=0.7234

  LinearSVC  C=3.0                    | F1=0.7234  Acc=0.8008


In [29]:

# ── Best model report ──────────────────────────────────────────────────────
best_label = max(results, key=lambda k: results[k][0])
f1_best, acc_best, y_pred_best = results[best_label]

print("\n" + "=" * 64)
print(f"BEST MODEL: {best_label}")
print("=" * 64)
print(f"Accuracy:      {acc_best:.4f}  ({acc_best * 100:.2f}%)")
print(f"F1 (macro):    {f1_best:.4f}")
print(f"F1 (weighted): {f1_score(y_test, y_pred_best, average='weighted'):.4f}")
print()
print("=== Per-class report ===")
print(classification_report(
    y_test, y_pred_best,
    target_names=list(le.classes_),
    digits=4,
))


print("\n=== All Results Summary (sorted by F1 macro) ===")
for label, (f1, acc, _) in sorted(results.items(), key=lambda x: -x[1][0]):
    print(f"  F1={f1:.4f}  Acc={acc:.4f}  {label}")



BEST MODEL: LinearSVC  C=0.5
Accuracy:      0.8110  (81.10%)
F1 (macro):    0.7519
F1 (weighted): 0.8060

=== Per-class report ===
                precision    recall  f1-score   support

           БАР     0.7228    0.5092    0.5975       379
           ОКР     0.7668    0.6782    0.7198       926
           ПРЛ     0.7534    0.7453    0.7493       742
     депрессия     0.8552    0.9241    0.8883      1739
тревожное р-во     0.8270    0.8915    0.8581      1180
    шизофрения     0.7586    0.6471    0.6984        34

      accuracy                         0.8110      5000
     macro avg     0.7807    0.7326    0.7519      5000
  weighted avg     0.8064    0.8110    0.8060      5000


=== All Results Summary (sorted by F1 macro) ===
  F1=0.7519  Acc=0.8110  LinearSVC  C=0.5
  F1=0.7450  Acc=0.8088  LinearSVC  C=0.3
  F1=0.7376  Acc=0.8084  LinearSVC  C=1.0
  F1=0.7357  Acc=0.8062  LinearSVC  C=1.5
  F1=0.7340  Acc=0.8040  LinearSVC  C=0.2
  F1=0.7294  Acc=0.8032  LinearSVC  C=2.0
  F